# 1. Project Overview

**Google Street View — Landmark Recognition** (Kaggle competition: [Google Landmark Recognition 2021](https://www.kaggle.com/competitions/landmark-recognition-2021))

This notebook is designed to run **directly inside a Kaggle Notebook**, using the competition dataset mounted under `/kaggle/input`. It does **not** require Google Colab, Google Drive, or downloading the ~105 GB dataset locally.

**Pipeline:**

```
Locate dataset on Kaggle
        ↓
Load train.csv
        ↓
Filter a manageable subset of landmark classes + undersample
        ↓
Stratified train / validation / test split (at the dataframe level)
        ↓
tf.data pipeline (images read from disk on demand, never fully loaded into RAM)
        ↓
Data augmentation (training data only)
        ↓
CNN from scratch (5 conv blocks → dense head)
        ↓
Class weights (used instead of image-duplication oversampling)
        ↓
Training with EarlyStopping / ReduceLROnPlateau / ModelCheckpoint
        ↓
Evaluation, classification report, confusion matrix
        ↓
Sample prediction visualization
        ↓
Save model to /kaggle/working
```

**Important design note — academic subset:** For computational feasibility, this academic project trains on a controlled subset of landmark classes rather than the entire competition dataset (which has 81,000+ classes and ~1.58M images). The subset size is controlled explicitly by `MIN_CLASS`, `MAX_CLASS`, and `UNDERSAMPLING_THRESHOLD` below, and the resulting class/image counts are printed so you always know exactly how much data was used.

# 2. Environment and GPU Check

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", gpus)

if gpus:
    print("GPU is available.")
else:
    print("WARNING: No GPU detected. Enable GPU from Kaggle Notebook Settings "
          "(right panel > Settings > Accelerator > GPU).")

# 3. Imports and Configuration

In [ ]:
import os
import gc
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------------
# Project configuration
# (change these to control how much data is used)
# -----------------------------
IMG_SIZE = 128

MIN_CLASS = 1
MAX_CLASS = 500

UNDERSAMPLING_THRESHOLD = 50     # max images kept per class
MIN_IMAGES_PER_CLASS = 5         # classes with fewer images than this are dropped
                                  # (needed so every class can appear in train/val/test)

BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-3

print("Configuration:")
print("  IMG_SIZE                =", IMG_SIZE)
print("  MIN_CLASS / MAX_CLASS    =", MIN_CLASS, "/", MAX_CLASS)
print("  UNDERSAMPLING_THRESHOLD  =", UNDERSAMPLING_THRESHOLD)
print("  MIN_IMAGES_PER_CLASS     =", MIN_IMAGES_PER_CLASS)
print("  BATCH_SIZE               =", BATCH_SIZE)
print("  EPOCHS (max, with EarlyStopping) =", EPOCHS)
print("  LEARNING_RATE            =", LEARNING_RATE)

# 4. Locate Kaggle Dataset

The dataset folder name can vary slightly depending on how the competition data was attached, so we search `/kaggle/input` instead of hardcoding a single path. If the dataset isn't found, this cell raises a clear error telling you to attach it via **Add Input**.

In [ ]:
INPUT_ROOT = "/kaggle/input"

possible_dirs = [
    os.path.join(INPUT_ROOT, "landmark-recognition-2021"),
    os.path.join(INPUT_ROOT, "google-landmark-recognition-2021"),
    os.path.join(INPUT_ROOT, "competitions", "landmark-recognition-2021"),
]

DATASET_DIR = None

# First check the known possible locations
for d in possible_dirs:
    if os.path.isdir(d):
        DATASET_DIR = d
        break

# Fallback: recursively search under /kaggle/input for train.csv
if DATASET_DIR is None and os.path.isdir(INPUT_ROOT):
    for root, dirs, files in os.walk(INPUT_ROOT):
        if "train.csv" in files:
            DATASET_DIR = root
            break

if DATASET_DIR is None:
    raise FileNotFoundError(
        "Could not locate the Google Landmark Recognition 2021 dataset under /kaggle/input.\n"
        "Please attach it to this notebook:\n"
        "  1. Click 'Add Input' in the right-hand panel.\n"
        "  2. Search for 'Google Landmark Recognition 2021' (competition data).\n"
        "  3. Attach it, then re-run this cell."
    )

TRAIN_CSV = os.path.join(DATASET_DIR, "train.csv")
TRAIN_DIR = os.path.join(DATASET_DIR, "train")

assert os.path.exists(TRAIN_CSV), f"train.csv not found at {TRAIN_CSV}"
assert os.path.isdir(TRAIN_DIR), f"Train image directory not found at {TRAIN_DIR}"

print("Dataset directory:    ", DATASET_DIR)
print("train.csv path:       ", TRAIN_CSV)
print("Train image directory:", TRAIN_DIR)
print("\nDataset successfully found.")

# 5. Load train.csv

We only keep the columns we actually need (`id`, `landmark_id`) and build an `img_path` column pointing at each image on disk. We do **not** load any image bytes here — this is just metadata.

In [ ]:
def build_img_path(image_id):
    # landmark-recognition-2021 convention: train/<id[0]>/<id[1]>/<id[2]>/<id>.jpg
    return os.path.join(TRAIN_DIR, image_id[0], image_id[1], image_id[2], image_id + ".jpg")

traindf = pd.read_csv(TRAIN_CSV, usecols=["id", "landmark_id"])
traindf["landmark_id"] = traindf["landmark_id"].astype(np.int32)
traindf["img_path"] = traindf["id"].apply(build_img_path)

landmark_unique = traindf["landmark_id"].nunique()

print("Number of images in the full dataset:", traindf.shape[0])
print("Number of different classes in the full dataset:", landmark_unique)
traindf.head()

# 6. Dataset Exploration

Basic statistics, plus a quick peek at a handful of raw images (loaded one at a time, not cached into memory).

In [ ]:
print("Min class:", traindf["landmark_id"].min())
print("Max class:", traindf["landmark_id"].max())
print("\nTop 5 most frequent classes:")
print(traindf["landmark_id"].value_counts().head())
print("\nClasses with only 1 image:", (traindf["landmark_id"].value_counts() == 1).sum())

In [ ]:
# Peek at a small, random sample of raw images (loaded individually -- not cached)
sample_rows = traindf.sample(n=10, random_state=SEED)

plt.figure(figsize=(20, 8))
for i, (_, row) in enumerate(sample_rows.iterrows()):
    img = tf.io.read_file(row["img_path"])
    img = tf.image.decode_jpeg(img, channels=3)
    plt.subplot(2, 5, i + 1)
    plt.imshow(img.numpy())
    plt.xticks([]); plt.yticks([])
    plt.xlabel(f"landmark_id: {row['landmark_id']}")
plt.tight_layout()
plt.show()

# 7. Class Filtering and Undersampling

We restrict to landmark classes between `MIN_CLASS` and `MAX_CLASS`, then cap each class at `UNDERSAMPLING_THRESHOLD` images (keeping all available images for classes with fewer). Classes that still have fewer than `MIN_IMAGES_PER_CLASS` images afterward are dropped entirely, since they can't safely be split into train/validation/test.

We use `.apply()` on the *filtered, already-small* dataframe (not the full 1.58M-row dataset) because each group needs a different sample size (`min(group_size, UNDERSAMPLING_THRESHOLD)`), which plain `.groupby().sample(n=...)` can't express with a single fixed `n`.

In [ ]:
traindf_filtered = traindf[
    (traindf["landmark_id"] >= MIN_CLASS) & (traindf["landmark_id"] <= MAX_CLASS)
].copy()

def _sample_group(group):
    n = min(len(group), UNDERSAMPLING_THRESHOLD)
    return group.sample(n=n, random_state=SEED)

traindf_s = (
    traindf_filtered
    .groupby("landmark_id", group_keys=False)
    .apply(_sample_group)
    .reset_index(drop=True)
)

# Drop classes that are still too small to split safely
class_counts = traindf_s["landmark_id"].value_counts()
valid_classes = class_counts[class_counts >= MIN_IMAGES_PER_CLASS].index
traindf_s = traindf_s[traindf_s["landmark_id"].isin(valid_classes)].reset_index(drop=True)

N_DATA = len(traindf_s)
N_CLASS_RAW = traindf_s["landmark_id"].nunique()

print(f"Images after filtering + undersampling: {N_DATA}")
print(f"Classes after filtering + undersampling: {N_CLASS_RAW}")
print("\nPer-class image count summary:")
print(traindf_s["landmark_id"].value_counts().describe())

In [ ]:
# Label-encode landmark_id -> contiguous 0..NUM_CLASSES-1 indices for the model
LE = LabelEncoder()
traindf_s["label_enc"] = LE.fit_transform(traindf_s["landmark_id"])
NUM_CLASSES = traindf_s["label_enc"].nunique()

print("Final NUM_CLASSES used by the model:", NUM_CLASSES)

# 8. Train / Validation / Test Split

Split is performed at the **dataframe level** (paths + labels only), before any images are loaded. The main train/temp split is stratified by class; the much smaller val/test split attempts stratification too, falling back to a plain random split if a class is too small to stratify (this only affects a handful of rare classes).

In [ ]:
train_df, temp_df = train_test_split(
    traindf_s,
    test_size=0.30,
    random_state=SEED,
    stratify=traindf_s["label_enc"],
)

try:
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_df["label_enc"],
    )
except ValueError:
    print("Note: a few classes were too small to stratify the val/test split; "
          "falling back to a plain random split for that step.")
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=SEED, shuffle=True
    )

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Number of classes:          ", NUM_CLASSES)
print("Number of training samples:  ", len(train_df))
print("Number of validation samples:", len(val_df))
print("Number of test samples:      ", len(test_df))

# 9. TensorFlow Data Pipeline

Images are read from disk **on demand**, decoded, resized and normalized inside a `tf.data.Dataset` — the full image set is never materialized as a NumPy array in RAM. Corrupted/unreadable images are skipped automatically rather than crashing the pipeline (this is a dataset-level skip, so individual bad filenames aren't logged — if you need that, validate files separately before training).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3, try_recover_truncated=True, acceptable_fraction=0.5)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def _skip_errors(ds):
    # Skip corrupted/unreadable images instead of crashing the whole pipeline
    if hasattr(ds, "ignore_errors"):
        return ds.ignore_errors()
    return ds.apply(tf.data.experimental.ignore_errors())

def make_dataset(df, training=False):
    paths = df["img_path"].values
    labels = df["label_enc"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    ds = _skip_errors(ds)

    if training:
        # Augmentation is applied only to the training split (see section 10)
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

print("Data pipeline functions defined. Datasets will be built after the augmentation layer (section 10).")

# 10. Data Augmentation

Uses current (non-deprecated) Keras preprocessing layers. Applied **only** to the training dataset — validation and test data pass through unaugmented.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomZoom(height_factor=(-0.2, 0.2)),
    layers.RandomContrast(0.4),
    layers.RandomTranslation(height_factor=(-0.2, 0.2), width_factor=(-0.2, 0.2)),
    layers.RandomRotation(factor=(-0.1, 0.1)),
], name="data_augmentation")

# Now build the actual datasets
train_dataset = make_dataset(train_df, training=True)
val_dataset = make_dataset(val_df, training=False)
test_dataset = make_dataset(test_df, training=False)

print("train_dataset, val_dataset, test_dataset built.")

# 11. CNN Model

Same CNN-from-scratch architecture as the original notebook (5 conv blocks of increasing filter size, each with BatchNorm + MaxPooling, followed by a dense head) — not swapped for a pretrained backbone. Updated to use `keras.Input` instead of the deprecated `input_shape=` argument on the first `Conv2D`, and `Adam` with a sensible learning rate instead of raw SGD at `lr=0.01`.

In [ ]:
# CNN hyperparameters
kernelSize = (3, 3)
paddingType = "same"
activationF = "relu"
poolSize = (2, 2)
stridesSize = (2, 2)
dropoutRate = 0.5
weightDecay = regularizers.L2(0.01)

def create_model(num_classes):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # Convolutional block 1
    x = layers.Conv2D(32, kernelSize, padding=paddingType, activation=activationF)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=poolSize, strides=stridesSize, padding=paddingType)(x)

    # Convolutional block 2
    x = layers.Conv2D(64, kernelSize, padding=paddingType, activation=activationF)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=poolSize, strides=stridesSize, padding=paddingType)(x)

    # Convolutional block 3
    x = layers.Conv2D(128, kernelSize, padding=paddingType, activation=activationF)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=poolSize, strides=stridesSize, padding=paddingType)(x)

    # Convolutional block 4
    x = layers.Conv2D(256, kernelSize, padding=paddingType, activation=activationF)(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=poolSize, strides=stridesSize, padding=paddingType)(x)

    # Classification head
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation=activationF, kernel_regularizer=weightDecay)(x)
    x = layers.Dropout(dropoutRate, seed=SEED)(x)
    x = layers.BatchNormalization()(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="landmark_cnn")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = create_model(NUM_CLASSES)
model.summary()

# 12. Class Weights

The dataset is imbalanced even after undersampling, so we compute class weights from the **training labels only** and pass them to `model.fit()`, instead of physically duplicating images (which the original notebook did via repeated `np.concatenate`). This gets the same rebalancing benefit without the memory cost.

In [ ]:
train_labels = train_df["label_enc"].values
present_classes = np.unique(train_labels)

weights_array = compute_class_weight(
    class_weight="balanced", classes=present_classes, y=train_labels
)
class_weights = dict(zip(present_classes.tolist(), weights_array.tolist()))

# Make sure every class index the model can output has a weight (defaults to 1.0)
for c in range(NUM_CLASSES):
    if c not in class_weights:
        class_weights[c] = 1.0

print("Number of classes with computed weights:", len(class_weights))
print("Example weights (first 5):", dict(list(class_weights.items())[:5]))

# 13. Training

`EPOCHS` is a *ceiling*, not a target — `EarlyStopping` will stop training as soon as validation loss stops improving for `patience` epochs (restoring the best weights seen). Model checkpoints are saved under `/kaggle/working/` (never `/kaggle/input/`, which is read-only) using the modern `.keras` format.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
    ),
    keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_landmark_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
    ),
]

start_time = time.time()

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
)

print("\nTraining duration: %.2f minutes" % ((time.time() - start_time) / 60))

# 14. Training Curves

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(history_df["loss"], label="Training")
plt.plot(history_df["val_loss"], label="Validation")
plt.title("Loss and Validation Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df["accuracy"], label="Training")
plt.plot(history_df["val_accuracy"], label="Validation")
plt.title("Accuracy and Validation Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.savefig("/kaggle/working/training_history.png")
plt.show()

print("Minimum validation loss: {:.4f} at epoch {}".format(
    history_df["val_loss"].min(), int(history_df["val_loss"].idxmin())
))
print("Maximum validation accuracy: {:.4f} at epoch {}".format(
    history_df["val_accuracy"].max(), int(history_df["val_accuracy"].idxmax())
))

# 15. Test Evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

# 16. Classification Report

Predictions are generated batch-by-batch from `test_dataset`; only the small `y_true` / `y_pred` label arrays are collected, never a giant array of test images.

In [ ]:
y_true = []
y_pred = []

for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(y_true, y_pred, zero_division=0))

# 17. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap="Blues", cbar=True)
plt.title("Confusion Matrix (label-encoded class indices)")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png")
plt.show()

# 18. Sample Predictions

Only a small number of images (12) are visualized. For each one, a couple of example images of the *predicted* class are loaded directly from disk (not pre-cached) so you can visually sanity-check the prediction.

In [ ]:
sample_batch_images, sample_batch_labels = next(iter(test_dataset))
sample_images = sample_batch_images[:12].numpy()
sample_true = sample_batch_labels[:12].numpy()

sample_preds = model.predict(sample_images, verbose=0)
sample_pred_labels = np.argmax(sample_preds, axis=1)
sample_confidence = np.max(sample_preds, axis=1)

true_landmark_ids = LE.inverse_transform(sample_true)
pred_landmark_ids = LE.inverse_transform(sample_pred_labels)

plt.figure(figsize=(20, 12))
for i in range(len(sample_images)):
    plt.subplot(3, 4, i + 1)
    plt.imshow(sample_images[i])
    plt.xticks([]); plt.yticks([])
    status = "correct" if true_landmark_ids[i] == pred_landmark_ids[i] else "wrong"
    plt.title(
        f"True: {true_landmark_ids[i]}\nPred: {pred_landmark_ids[i]} ({status})\nConf: {sample_confidence[i]:.2f}",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

In [ ]:
# Show a couple of reference images for each predicted class (loaded from disk, not cached)
for i in range(len(sample_images)):
    predicted_class_id = pred_landmark_ids[i]
    examples = train_df[train_df["landmark_id"] == predicted_class_id].head(3)

    if len(examples) == 0:
        continue

    plt.figure(figsize=(9, 3))
    for j, (_, row) in enumerate(examples.iterrows()):
        img = tf.io.read_file(row["img_path"])
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]) / 255.0
        plt.subplot(1, len(examples), j + 1)
        plt.imshow(img.numpy())
        plt.xticks([]); plt.yticks([])
    plt.suptitle(f"Test image #{i} -> example images of predicted class {predicted_class_id}")
    plt.show()

# 19. Save Model

Everything is written to `/kaggle/working/` (writable); `/kaggle/input/` is read-only and never modified.

In [ ]:
final_model_path = "/kaggle/working/best_landmark_model.keras"
model.save(final_model_path)  # ModelCheckpoint already saved the best-val-accuracy version during training

print("Model saved to:", final_model_path)
print("Training history plot saved to: /kaggle/working/training_history.png")
print("Confusion matrix plot saved to: /kaggle/working/confusion_matrix.png")

print("\nFiles currently in /kaggle/working:")
for f in os.listdir("/kaggle/working"):
    print(" -", f)

# 20. Final Results and Conclusion

This notebook trained a from-scratch CNN on a controlled subset of the Google Landmark Recognition 2021 dataset (classes `MIN_CLASS`–`MAX_CLASS`, up to `UNDERSAMPLING_THRESHOLD` images per class), using a memory-efficient `tf.data` pipeline that reads images from disk on demand rather than loading the full dataset into RAM.

**Key results** (see printed output above for this run's actual numbers):
- Final test loss / test accuracy — see Section 15
- Per-class precision / recall / F1 — see Section 16
- Training/validation curves and confusion matrix are saved as PNGs in `/kaggle/working/`

**Possible next steps:** increase `MAX_CLASS` / `UNDERSAMPLING_THRESHOLD` if more compute/time is available, tune the learning rate or add more conv blocks, or try a pretrained backbone (not done here, to keep the original CNN-from-scratch methodology intact).